In [ ]:
import pandas as pd
import re
import requests
import io
import gzip

In [ ]:
import matplotlib.pyplot as plt
import tqdm as notebook_tqdm
from datetime import date
import seaborn as sns
import pandas as pd
import scanpy as sc
import numpy as np
import scipy
import sys
import os
import re
import harmonypy as hm
import palantir
import matplotlib
import matplotlib as mpl


In [ ]:
# import src.visualization.scBasic as kvis

f"Last execution: {date.today()}"

print(np.__version__)
print(sc.__version__)
print(sys.version)

# -- Set base directory
output_dir = '/nfs/team292/rs40/projects/Aneuploid_screen_v2/processed_data/3_scanpy_integration'
os.makedirs(output_dir, exist_ok=True)

pd.set_option("display.max_columns", 50)

# -- Input datafiles
files = {}

In [ ]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="Variable names are not unique. To make them unique, call `.var_names_make_unique`."
)

In [ ]:
np.random.seed(0)

## Scanpy setting

In [ ]:
plt.rcParams.update({
    "figure.figsize": (3, 3),    # Default figure size
    "figure.dpi": 300,           # High resolution
    "font.size": 7,              # Global font size (fallback)
    "axes.titlesize": 7,        # Title size (e.g., 'Leiden')
    "axes.labelsize": 7,         # Axis labels (e.g., 'UMAP1')
    "xtick.labelsize": 7,        # Tick numbers on X axis & Colorbars
    "ytick.labelsize": 7,        # Tick numbers on Y axis & Colorbars
    "legend.fontsize": 7,        # Legend text
    "lines.markersize": 1,       # Dot size in legends
    "axes.spines.top": False,    # Remove top border globally
    "axes.spines.right": False   # Remove right border globally
})


sc.settings.figdir = output_dir

sc.set_figure_params(
    scanpy=True,           # Use Scanpy's opinionated style defaults
    dpi=300,               # High resolution for publication
    dpi_save=300,          # Resolution for saved files
    frameon=True,         # Remove box around plots (cleaner)
    vector_friendly=False,  # usage for PDF/SVG editors (Illustrator)
    fontsize=7,            # Set the base font size (very small for 3-inch figures)
    figsize=(3, 3),        # Set default figure size
    #facecolor=None,     # Ensure background is white (not transparent)
    format='pdf'           # Default save format
)

plt.rcParams['xtick.labelsize'] = 7
plt.rcParams['ytick.labelsize'] = 7

# This is the critical setting for Adobe Illustrator
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # ships with Matplotlib


In [ ]:
sample_colors = {
    "T1_control":  "#C5D659",  # light yellow (Control)
    "T1_rev":      "#C89CD9",  # light lavender (Reversine)

    "T2_control":  "#63B34F",  # green (Control)
    "T2_rev":      "#9C60B7",  # medium purple (Reversine)
    "T2_mix":      "#F6CF63",  # yellow (Mix)

    "T3_naive_rev":  "#5A2C6C",  # dark purple (Reversine)
    "T3_control":  "#3F8E8A",  # teal (Control)
    "T3_mix_good":    "#E76E35",  # bright orange (Mix)
    "T3_mix_bad":    "#E7A03E",  # darker orange-brown (Mix)
    "T3_rev":      "#3F3C96",  # deep blue-purple (Reversine)
}

timepoint_colors = {
    "day1": "#F2F08A",  # yellow
    "day4": "#4E9CA6",  # teal
    "day6": "#5B3C7F",  # purple
}

celltype_colors = {
    "naive EPI":    "#F6CF63",  # warm yellow
    "blastoid EPI": "#E79033",  # orange
    "TE":       "#3F77C5",  # blue
    "unspecified":       "#63B34F",  # green
}

state_colors = {
    "Blastoid_Gd": "#3F56A8",   # deep blue
    "Blastoid_Bd": "#3F8E8A",   # teal
    "Intermediate": "#E79033",   # orange
    "Naïve":        "#F6CF63",   # pale yellow
}

condition_colors = {
    "Control":   "#63B34F",  # dark grey
    "Mosaic":    "#E79033",  # orange
    "Reversine": "#9C60B7",  # purple
}

condition_colors2 = {
    "Control":   "#6DAE4E",  # dark grey
    "Mosaic":    "#F2B84B",  # orange
    "Reversine": "#C23A75",  # purple
}

celltype_colors = {
    "naive EPI":    "#F8B949",  # warm yellow
    "blastoid EPI": "#E79033",  # orange
    "TE":       "#3F77C5",  # blue
    "unspecified":       "#676767",  # ggrey
}


from matplotlib.colors import LinearSegmentedColormap
grey_to_pink = LinearSegmentedColormap.from_list("grey_to_pink", ["#D3D3D3", "#B54574"])

## Load data

In [ ]:
samples = ['Embryo-Imp15265938', 
           'Embryo-Imp15265939',
           'Embryo-Imp15265940',
           'Embryo-Imp15265941',
           'Embryo-Imp15265942',
           'Embryo-Imp15265943',
           'Embryo-Imp15265944',
           'Embryo-Imp15265945',
           'Embryo-Imp15265946',
           'Embryo-Imp15265947']

In [ ]:
adatas = []
metas = []

for sample in samples:
        adata = sc.read_10x_h5('/lustre/scratch126/cellgen/vento/rs40/from_iRODs/cellranger900_count_49831_' + sample + '_GRCh38-2024-A/filtered_feature_bc_matrix.h5')
        adata.var_names_make_unique()
        adata.obs['dataset']=sample
        adata.obs.index = adata.obs.index + "_" + adata.obs['dataset']
        adatas.append(adata)
        adata.obs["cell.ID_"]= adata.obs.index 
        
        meta = pd.read_csv('/nfs/team292/rs40/projects/Aneuploid_screen_v2/processed_data/2_scanpy/' + sample +'/'+ sample + '.csv',sep=',', index_col=0)
        meta.index = meta.index + "_" +sample
        metas.append(meta)
        
        

In [ ]:
adata = sc.concat(adatas, axis = 0, join = 'outer')
meta_df = pd.concat(metas, axis=0)

In [ ]:
adata.obs = adata.obs.join(meta_df, how="left")

In [ ]:
sample_list = pd.read_csv('/nfs/team292/rs40/projects/Aneuploid_screen_v2/notebooks/sample_list.csv', index_col=0)

In [ ]:
adata.obs = adata.obs.merge(
    sample_list,
    on="dataset",   # use 'dataset' column in BOTH dataframes
    how="left",
    sort=False,
)

## Initial threshold

In [ ]:
#Save the raw counts as a layer
adata.layers["counts"] = adata.X.copy()

#very minimal filter to detect likely empty droplet, noise
#filter genes detected in less than 5 cells
sc.pp.filter_genes(adata, min_cells=5)

#filter cells with less than 100 genes
sc.pp.filter_cells(adata,min_genes =100)
adata

# define mitochondrial genes (note that this may start with "mt-" or "MT-" depending on dataset)
adata.var["mt"] = adata.var_names.str.startswith("MT-") 

#define ribosomal genes. this is not necessary for QC, but maybe biologically intresting?
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))

#calculate QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
sc.pp.calculate_qc_metrics(adata, qc_vars=['ribo'], percent_top=None, log1p=False, inplace=True)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (5, 3), 
    "figure.dpi": (600)
}):
    sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
                 jitter=0.4, groupby = 'sample', rotation= 90, size = 0.5, save = "QC_violin.pdf")   

In [ ]:
#filter based on plots above
min_genes = 2000
adata = adata[adata.obs['n_genes_by_counts'] >= min_genes]
adata = adata[adata.obs['total_counts'] <= 100000]
adata = adata[adata.obs['pct_counts_mt'] <= 20]

#count number of samples
sample_counts = adata.obs['dataset'].value_counts()[adata.obs['dataset'].unique()]

# Get default Matplotlib color cycle
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# Plot the counts
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
     "axes.grid":False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
   "savefig.dpi" :(300)
}):
    plt.figure(figsize=(3, 3))
    sample_counts.plot(kind='bar', edgecolor='none', color = colors)
    plt.ylabel('Number of Cells')
    plt.title('Number of Cells per Sample')
    plt.xticks(rotation=90)
    plt.savefig(output_dir +'/number_of_cells_per_sample.pdf', dpi=300, bbox_inches='tight')
    plt.show()

## Normalise, Cluster

In [ ]:
#normalise
sc.pp.normalize_total(adata,
                         target_sum = 1e4)

sc.pp.log1p(adata)

#score cell cycle genes
s_genes = ['MCM5','PCNA','TYMS','FEN1','MCM7','MCM4','RRM1','UNG','GINS2','MCM6','CDCA7','DTL','PRIM1',
           'UHRF1','CENPU','HELLS','RFC2','POLR1B','NASP','RAD51AP1','GMNN','WDR76','SLBP','CCNE2','UBR7',
           'POLD3','MSH2','ATAD2','RAD51','RRM2','CDC45','CDC6','EXO1','TIPIN','DSCC1','BLM','CASP8AP2',
           'USP1','CLSPN','POLA1','CHAF1B','MRPL36','E2F8']
g2m_genes = ['HMGB2','CDK1','NUSAP1','UBE2C','BIRC5','TPX2','TOP2A','NDC80','CKS2','NUF2','CKS1B',
             'MKI67','TMPO','CENPF','TACC3','PIMREG','SMC4','CCNB2','CKAP2L','CKAP2','AURKB','BUB1',
             'KIF11','ANP32E','TUBB4B','GTSE1','KIF20B','HJURP','CDCA3','JPT1','CDC20','TTK','CDC25C',
             'KIF2C','RANGAP1','NCAPD2','DLGAP5','CDCA2','CDCA8','ECT2','KIF23','HMMR','AURKA','PSRC1',
             'ANLN','LBR','CKAP5','CENPE','CTCF','NEK2','G2E3','GAS2L3','CBX5','CENPA']
cell_cycle_genes = s_genes + g2m_genes
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)


#score cell cycle genes
stress_genes = [x.strip() for x in open('/nfs/team292/rs40/projects/Aneuploid_screen/notebooks/integration/stress_mmu.tsv')]
sc.tl.score_genes(adata, score_name='stress_score_vandenBrink', gene_list=stress_genes)


In [ ]:
# Clustering
sc.pp.highly_variable_genes(adata,
                            flavor = 'seurat_v3',
                            n_top_genes = 4000,
                             layer='counts' ,
                            subset = False, batch_key= 'dataset')

sc.pl.highly_variable_genes(adata,log = True)


#filter highly variable genes, do PCA
bdata = adata[:, adata.var.highly_variable]

#scale the variance, 0 centre, 10 max
sc.pp.scale(bdata,
            max_value = 10)

#compute PCA
sc.tl.pca(bdata,
          n_comps = 100,
          svd_solver = 'arpack')

# Tranfer PCA values from bdata to adata

#fill NaNs with False so that subsetting to HVGs is possible
adata.var['highly_variable'].fillna(value = False,
                                    inplace = True)

adata.obsm['X_pca'] = bdata.obsm['X_pca'].copy()
adata.uns['pca'] = bdata.uns['pca'].copy()

adata.varm['PCs'] = np.zeros(shape=(adata.n_vars,
                                    100))
adata.varm['PCs'][adata.var['highly_variable']] = bdata.varm['PCs']

sc.pl.pca_variance_ratio(adata,
                         n_pcs = 100,
                         log = True)

In [ ]:
n_pcs = 30 # based on rough knee point
n_neighbors = 15

#compute leiden clusters
sc.pp.neighbors(adata,
                n_pcs = n_pcs,
                n_neighbors = n_neighbors,
                random_state=0)

sc.tl.umap(adata, min_dist=0.5, spread=1.0, random_state=0)

sc.tl.leiden(adata,
             resolution = 1)

# -- Modify name to UMAP
adata.obsm[f"umap_without_integration"] = adata.obsm['X_umap']


# random permutation of cells
np.random.seed(0)
idx = np.random.permutation(adata.n_obs)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
    'axes.titlesize':8,
     "legend.fontsize": 8,
    'axes.labelsize' :8
}):
    QC_plot = sc.pl.umap(adata[idx, :],
           color = ["leiden", 'sample',  'sample_celltype', 'State', 'timepoint', "Condition", "n_genes_by_counts",  "pct_counts_mt",  'scrublet_cluster_score'],
           wspace = 0.3,
           ncols = 3,
           alpha = 0.75,
           size = 5,
           save = "_preintegration_QC.pdf"
#            legend_loc = 'on data',
          )
QC_plot

In [ ]:

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
    'axes.titlesize':8,
     "legend.fontsize": 8,
    'axes.labelsize' :8
}):
    ax = sc.pl.umap(adata, size= 20, show=False)
    sc.pl.umap(
        adata[adata.obs['scrublet_cluster_score'] > 0.3],
        color=["leiden"],
        ax=ax, 
        size= 10
    # palette=celltype_colors, 
       # save = "_annotation_" +sample +".pdf"
    )

## Harmony

In [ ]:
# --- Integration with Harmony ---

harmony_batch = ["dataset"]
thetas = [0, 1, 2, 3]   # explore a few values

# Make sure batch covariates are categorical
for c in harmony_batch:
    if c in adata.obs:
        adata.obs[c] = adata.obs[c].astype("category")

np.random.seed(1)

# Use existing PCA (cells × PCs) as input to Harmony
pca_in = adata.obsm["X_pca"][:, :n_pcs]

# 1) Run Harmony for each theta and store corrected PCs
for theta in thetas:
    ho = hm.run_harmony(
        pca_in.T,        # features × samples
        adata.obs,       # metadata
        harmony_batch,
        theta=theta,
        max_iter_harmony=30,
    )

    # store corrected PCs (cells × PCs) for this theta
    adata.obsm[f"X_pca_harmony_theta{theta}"] = ho.Z_corr.T
    


In [ ]:
## plot 

np.random.seed(0)
idx = np.random.permutation(adata.n_obs)

for theta in thetas:

    # 2) Build graph/UMAP using this representation for QC
    sc.pp.neighbors(
        adata,
        use_rep=f"X_pca_harmony_theta{theta}",
        n_neighbors=15,
        random_state=0,
    )
    sc.tl.leiden(
        adata,
        resolution=1,
        key_added=f"leiden_theta{theta}"
    )
    sc.tl.umap(
        adata,
        min_dist=0.5,
        spread=1.0,
        random_state=0,
    )

    # 3) QC UMAP panels for this theta
    sc.pl.umap(
        adata[idx, :],
        color=[
            f"leiden_theta{theta}" ,"pct_counts_mt",   "n_genes_by_counts", 'scrublet_cluster_score',
            "sample","timepoint",  'sample_celltype', "Condition"
        ],
        wspace=0.4,
        legend_fontsize="x-small",
        frameon=False,
        size=10,
        ncols=4,
        alpha=0.8,
        # save=f"_QCplot_integration_theta{theta}.pdf",
    )

I decided to use the non-integrated version for now.

In [ ]:
# --- Choose a final theta and build the official integrated object ---

best_theta = 0 

# Use the chosen Harmony PCs as the main integrated embedding
adata.obsm["X_pca_harmony"] = adata.obsm[f"X_pca_harmony_theta{best_theta}"].copy()

# # Neighbors / UMAP / clustering on the final integrated representation
# sc.pp.neighbors(
#     adata,
#     use_rep="X_pca_harmony",
#     n_neighbors=15,
#     random_state=0,
# )
# sc.tl.umap(adata, min_dist=0.2, spread=1.0, random_state=0)
# sc.tl.leiden(adata, resolution=1.3, key_added="leiden_harmony")

# # Save the final integrated AnnData
# adata.write_h5ad(os.path.join(output_dir, f"adata_harmony_theta{best_theta}.h5ad"))


In [ ]:
n_pcs = 30 # based on rough knee point
n_neighbors = 15

#compute leiden clusters
sc.pp.neighbors(adata,
                n_pcs = n_pcs,
                n_neighbors = n_neighbors,
                random_state=0)

sc.tl.umap(adata, min_dist=0.5, spread=1.0, random_state=0)

sc.tl.leiden(adata,
             resolution = 1)

# -- Modify name to UMAP
adata.obsm[f"umap_without_integration"] = adata.obsm['X_umap']


# random permutation of cells
np.random.seed(0)
idx = np.random.permutation(adata.n_obs)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
    'axes.titlesize':8,
     "legend.fontsize": 8,
    'axes.labelsize' :8
}):
    QC_plot = sc.pl.umap(adata[idx, :],
           color = ["leiden", 'sample',  'sample_celltype', 'State', 'timepoint', "Condition", "n_genes_by_counts",  "pct_counts_mt",  'scrublet_cluster_score'],
           wspace = 0.5,
           ncols = 3,
           alpha = 0.75,
           size = 5,
           save = "_preintegration_QC.pdf"
#            legend_loc = 'on data',
          )
QC_plot

## Marker genes expression 

In [ ]:
marker_genes = {
    "Naive hPSC": {"KLF17", "DPPA5", "DNMT3L", "GATA6", "TBX3", "IL6ST", "DPPA3", "KLF5", "HORMAD1", "KHDC3L", "ALPP", "ZNF729"},
    "Primed hPSC": {"CD24", "ZIC2", "SFRP2"},
    "ICM": {"IFITM1", "GDF3", "ARGFX", "ATG2A", "MAGEA4", "ESRRB", "FGF1", "PRSS3", "LAMA4", "CCR8", "EPHA4", "GSC"},
    "Trophectoderm": {"GATA2", "GATA3", "CDX2", "TFAP2C", "KRT7", "KRT18", "OVOL1"},
    "Mural TE": {"HAND1"},
    "Polar TE": {"SDC1", "GCM1", "OVOL1", "CCR7", "CYP19A1", "DLX5", "MUC15"},
    "Epiblast": {"POU5F1", "GDF3", "ARGFX", "NANOG", "SOX2", "KLF17", "TDGF1"},
    "Early Epiblast": {"DPPA5", "ARGFX", "SOX2"},
    "Late Epiblast": {"FGF2", "TDGF1", "NODAL"},
    "Hypoblast": {"PDGFRA", "GATA6", "GATA4", "SOX17", "COL4A1", "APOA1", "RSPO3"},
    "Amnion": {"GABRP", "ISL1","GABPR", "MSX2", "RARRES2"},
    "Meso": {"APLNR", "CRABP2", "UBB", "FAM162A", "MIF"}
}


In [ ]:
filtered_marker_genes = {}
skipped_genes = {}

for category, genes in marker_genes.items():
    # Find genes that exist in the dataset
    existing_genes = [gene for gene in genes if gene in adata.var_names]

    # Track skipped genes
    skipped_for_category = [gene for gene in genes if gene not in adata.var_names]

    # Only add category if there are existing genes
    if existing_genes:
        filtered_marker_genes[category] = existing_genes

    # Store skipped genes if any
    if skipped_for_category:
        skipped_genes[category] = skipped_for_category
        
# Print skipped genes
print("Skipped Genes:")
for category, genes in skipped_genes.items():
    print(f"{category}: {genes}")

In [ ]:
sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))

sc.pl.dotplot(
    adata,
    groupby="leiden",
    var_names=filtered_marker_genes,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "marker.pdf")

In [ ]:
# 1) Run DE for all clusters vs rest
sc.tl.rank_genes_groups(
    adata,
    groupby= "leiden",
    method="wilcoxon"
)

In [ ]:
# 2) Heatmap of top N DE genes per cluster
sc.pl.rank_genes_groups_heatmap(
    adata,
    n_genes=6,        # number of top genes per cluster
    standard_scale="var",  # z-score per gene
    #swap_axes=True,        # genes on y, clusters on x (often nicer)
    show_gene_labels=True,
    dendrogram=False,  # standard scale: normalize each gene to range from 0 to 1
    save = "_DE_heatmap.pdf"
)

In [ ]:
#based on the visual inspection of UMAP, annotate clusters:
cl_annotation = {str(i): "unspecified" for i in range(0,30)}  # Initialize all as "unknown"

# Assign specific values
for key in ["21", "22", "12", "17"]:
    cl_annotation[key] = "doublet"
            
for key in ["16"]:
    cl_annotation[key] = "MEFs"

for key in ["13", "18", "15", "14", "20"]:
    cl_annotation[key] = "low quality"
    
for key in ["5"]:
    cl_annotation[key] = "TE"
    
for key in []:
    cl_annotation[key] = "polarTE"
    
for key in [ "10","0"]:
    cl_annotation[key] = "EPI"

for key in []:
    cl_annotation[key] = "unknown"

adata.obs["integrated_celltype"] = adata.obs.leiden.map(cl_annotation)

In [ ]:
#visualise the clusters so far
#plot for QC
sc.settings.set_figure_params(dpi = 100)
sc.set_figure_params(figsize=(3, 3))

g=sc.pl.umap(adata, color=["integrated_celltype","leiden"],
           #legend_loc="on data",
           ncols = 4,
           legend_fontsize = 'xx-small',
           size = 10,
           alpha = 0.75,
           wspace=0.5,
           save = "_integrated_celltype.pdf")

g

## Save

In [ ]:
#save 
adata.obs['scrublet_prediction'] = adata.obs['scrublet_prediction'].astype(str)
adata.write_h5ad(os.path.join(output_dir, "integrated_adata.h5ad"))

# Analysis of filtered cells

In [ ]:
adata = sc.read_h5ad(os.path.join(output_dir, "integrated_adata.h5ad"))

In [ ]:
#by annotation
adata_sub = adata[~adata.obs["integrated_celltype"].isin(["doublet",  "low quality", "MEFs"])].copy()

#cluster score threshold
adata_sub = adata_sub[adata_sub.obs["scrublet_cluster_score"] <= 0.3].copy()

## HVG, PCA, Cluster

In [ ]:
# Highly variable genes
sc.pp.highly_variable_genes(
    adata_sub,
    flavor = 'seurat_v3',
    n_top_genes = 4000,
    subset = False, 
    batch_key= 'dataset',
    layer='counts' 
)

bdata_sub = adata_sub[:, adata_sub.var["highly_variable"]].copy()

#scale the variance, 0 centre, 10 max
sc.pp.scale(bdata_sub,
            max_value = 10)

#compute PCA
sc.tl.pca(bdata_sub,
          n_comps = 100,
          svd_solver = 'arpack')


#fill NaNs with False so that subsetting to HVGs is possible
bdata_sub.var['highly_variable'].fillna(value = False,
                                    inplace = True)

adata_sub.obsm['X_pca'] = bdata_sub.obsm['X_pca'].copy()
adata_sub.uns['pca'] = bdata_sub.uns['pca'].copy()

adata_sub.varm['PCs'] = np.zeros(shape=(adata_sub.n_vars,
                                    100))
adata_sub.varm['PCs'][adata_sub.var['highly_variable']] = bdata_sub.varm['PCs']

sc.pl.pca_variance_ratio(adata_sub,
                         n_pcs = 100,
                         log = True)


In [ ]:
# Neighbors / Leiden / UMAP on final rep
sc.pp.neighbors(adata_sub, n_pcs = 30, n_neighbors=30, random_state=0)

sc.tl.umap(adata_sub, min_dist=0.5, spread=1.0)
sc.tl.leiden(adata_sub, resolution=1, key_added="leiden")


In [ ]:
idx = np.random.permutation(adata_sub.n_obs)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
    'axes.titlesize':8,
     "legend.fontsize": 8,
    'axes.labelsize' :8
}):
    QC_plot = sc.pl.umap(adata_sub[idx, :],
           color = ['sample',  'sample_celltype', 'State', 'timepoint', "Condition", "n_genes_by_counts",  "pct_counts_mt",  'scrublet_cluster_score'],
           wspace = 0.4,
           ncols = 4,
           alpha = 0.75,
           size = 5,
           save = "_subset_integration_QC.pdf",     # any continuous obs
    color_map="viridis"
#            legend_loc = 'on data',
          )
QC_plot

## Marker genes

In [ ]:
marker_genes = {
    "Naive hPSC": {"KLF17", "DPPA5", "DNMT3L", "GATA6", "TBX3", "IL6ST", "DPPA3", "KLF5", "HORMAD1", "KHDC3L", "ALPP", "ZNF729"},
    "Primed hPSC": {"CD24", "ZIC2", "SFRP2"},
    "ICM": {"IFITM1", "GDF3", "ARGFX", "ATG2A", "MAGEA4", "ESRRB", "FGF1", "PRSS3", "LAMA4", "CCR8", "EPHA4", "GSC"},
    "Trophectoderm": {"GATA2", "GATA3", "CDX2", "TFAP2C", "KRT7", "KRT18", "OVOL1"},
    "Mural TE": {"HAND1"},
    "Polar TE": {"SDC1", "GCM1", "OVOL1", "CCR7", "CYP19A1", "DLX5", "MUC15"},
    "Epiblast": {"POU5F1", "GDF3", "ARGFX", "NANOG", "SOX2", "KLF17", "TDGF1"},
    "Early Epiblast": {"DPPA5", "ARGFX", "SOX2"},
    "Late Epiblast": {"FGF2", "TDGF1", "NODAL"},
    "Hypoblast": {"PDGFRA", "GATA6", "GATA4", "SOX17", "COL4A1", "APOA1", "RSPO3"},
    "Amnion": {"GABRP", "ISL1"},
    "Meso": {"APLNR", "CRABP2"}
}


In [ ]:
filtered_marker_genes = {}
skipped_genes = {}

for category, genes in marker_genes.items():
    # Find genes that exist in the dataset
    existing_genes = [gene for gene in genes if gene in adata_sub.var_names]

    # Track skipped genes
    skipped_for_category = [gene for gene in genes if gene not in adata_sub.var_names]

    # Only add category if there are existing genes
    if existing_genes:
        filtered_marker_genes[category] = existing_genes

    # Store skipped genes if any
    if skipped_for_category:
        skipped_genes[category] = skipped_for_category
        
# Print skipped genes
print("Skipped Genes:")
for category, genes in skipped_genes.items():
    print(f"{category}: {genes}")

In [ ]:
sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))

sc.pl.dotplot(
    adata_sub,
    groupby="leiden",
    var_names=filtered_marker_genes,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "submarker.pdf")

In [ ]:
#based on the visual inspection of UMAP, annotate clusters:
cl_annotation = {str(i): "unspecified" for i in range(0,30)}  # Initialize all as "unknown"

# Assign specific values
for key in []:
    cl_annotation[key] = "doublet"
            
for key in []:
    cl_annotation[key] = "MEFs"

for key in []:
    cl_annotation[key] = "low quality"
    
for key in ["4"]:
    cl_annotation[key] = "TE"
    
for key in []:
    cl_annotation[key] = "polarTE"
    
for key in [ "0","14"]:
    cl_annotation[key] = "naive EPI"

for key in ["9"]:
    cl_annotation[key] = "blastoid EPI"

adata_sub.obs["integrated_celltype"] = adata_sub.obs.leiden.map(cl_annotation)

In [ ]:
#visualise the clusters so far
#plot for QC
sc.settings.set_figure_params(dpi = 100)
sc.set_figure_params(figsize=(3, 3))

g=sc.pl.umap(adata_sub, color=["integrated_celltype","sample"],
           #legend_loc="on data",
           ncols = 4,
           legend_fontsize = 'xx-small',
           size = 10,
           alpha = 0.75,
           wspace=0.5,
           save = "_sub_integrated_celltype.pdf")

g

## Plots

In [ ]:
title = "celltype"
idx = np.random.permutation(adata_sub.n_obs)


with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (1.5, 1.5),
    "figure.dpi": 300,
    "axes.titlesize": 7,
    "legend.fontsize": 7,
    "axes.labelsize": 7,
}):
    sc.pl.umap(
        adata_sub[idx, :],
        color="integrated_celltype",
        size=1,
        #palette=sample_colors,
        #edgecolor="none",
        linewidth=0.5, 
        save = title + "_annotation_"+".pdf"
    )



In [ ]:
title = "samples"
idx = np.random.permutation(adata_sub.n_obs)

    
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (1.5, 1.5),
    "figure.dpi": 300,
    "axes.titlesize": 7,
    "legend.fontsize": 7,
    "axes.labelsize": 7,
}):
    sc.pl.umap(
        adata_sub[idx, :],
        color="sample",
        size=1,
        #palette=sample_colors,
        #edgecolor="none",
        linewidth=0.5,
        palette=sample_colors,
        save = title + "_annotation_"+".pdf"
    )
    
    

In [ ]:
timpoint = "day1"
idx = np.random.permutation(adata_sub[adata_sub.obs['timepoint'] == timpoint].n_obs)

#sc.tl.umap(adata, min_dist=min_dist, spread=spread,random_state=0)
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
    'axes.titlesize':8,
     "legend.fontsize": 8,
    'axes.labelsize' :8
}):
    ax = sc.pl.umap(adata_sub, size= 10, show=False)
    sc.pl.umap(
        adata_sub[adata_sub.obs['timepoint'] == timpoint][idx, :],
        color=["sample"],
        ax=ax, 
        size= 10,          
    palette=sample_colors, 
        save = timpoint + "_annotation_"+".pdf"
    )

In [ ]:
timpoint = "day4"
idx = np.random.permutation(adata_sub[adata_sub.obs['timepoint'] == timpoint].n_obs)

#sc.tl.umap(adata, min_dist=min_dist, spread=spread,random_state=0)
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
    'axes.titlesize':8,
     "legend.fontsize": 8,
    'axes.labelsize' :8
}):
    ax = sc.pl.umap(adata_sub, size= 10, show=False)
    sc.pl.umap(
        adata_sub[adata_sub.obs['timepoint'] == timpoint][idx, :],
        color=["sample"],
        ax=ax, 
        size= 10,          
    palette=sample_colors, 
        save = timpoint + "_annotation_"+".pdf"
    )

In [ ]:
timpoint = "day6"
idx = np.random.permutation(adata_sub[adata_sub.obs['timepoint'] == timpoint].n_obs)

#sc.tl.umap(adata, min_dist=min_dist, spread=spread,random_state=0)
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
    'axes.titlesize':8,
     "legend.fontsize": 8,
    'axes.labelsize' :8
}):
    ax = sc.pl.umap(adata_sub, size= 10, show=False)
    sc.pl.umap(
        adata_sub[adata_sub.obs['timepoint'] == timpoint][idx, :],
        color=["sample"],
        ax=ax, 
        size= 10,          
    palette=sample_colors,
        alpha=0.7,
        save = timpoint + "_annotation_"+".pdf"
    )

In [ ]:
marker_genes = {
   # "Naive hPSC": {"KLF17", "DPPA5", "IL6ST", "ZNF729"},
    "ICM": {"ATG2A",  "ESRRB", "LAMA4", "CCR8", "EPHA4"},
    "Epiblast": {"POU5F1", "DPPA5", "GDF3",  "NANOG", "SOX2", "KLF17", "TDGF1"},
    #"Early Epiblast": {"DPPA5", "ARGFX", "SOX2"},
    #"Late Epiblast": {"FGF2", "TDGF1", "NODAL"},
    "Trophectoderm": {"GATA3", "CDX2", "TFAP2C", "KRT7"},
    "Hypoblast": {"PDGFRA", "GATA6", "GATA4", "SOX17"},
    "Post-Imp": {"GABRP", "ISL1", "APLNR", "CRABP2"},
}


In [ ]:
sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))

sc.pl.dotplot(
    adata_sub,
    groupby="integrated_celltype",
    var_names=marker_genes,  cmap=grey_to_pink,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "celltype_submarker.pdf")

In [ ]:
sc.pl.umap(adata_sub, color=["State"],
           #legend_loc="on data",
           ncols = 4,
           legend_fontsize = 'xx-small',
           size = 10,
           alpha = 0.75,
           wspace=0.5, palette = state_colors,
    show = False)
sc.pl.umap(adata_sub, color=["timepoint"],
           #legend_loc="on data",
           ncols = 4,
           legend_fontsize = 'xx-small',
           size = 10,
           alpha = 0.75,
           wspace=0.5, palette = timepoint_colors,
    show = False)

sc.pl.umap(adata_sub, color=["Condition"],
           #legend_loc="on data",
           ncols = 4,
           legend_fontsize = 'xx-small',
           size = 10,
           alpha = 0.75,
           wspace=0.5, palette = condition_colors,
    show = False)

In [ ]:
idx = np.random.permutation(adata_sub.n_obs)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (2, 2), 
    "figure.dpi": (300),
    'axes.titlesize':8,
    "legend.fontsize": 8,
    'axes.labelsize' :8,
    # Reduce the size of the numbers on the axis and colorbar
    "xtick.labelsize": 8,          
    "ytick.labelsize": 8
}):
    QC_plot = sc.pl.umap(adata_sub[idx, :],
           color = ["n_genes_by_counts",  "pct_counts_mt", 'scrublet_cluster_score',  'timepoint', "Condition", 'State'],
           wspace = 0.5,
           ncols = 6,
           alpha = 0.75,
           size = 5,
           save = "_subset_integration_QC.pdf",     
    color_map='GnBu'
#            legend_loc = 'on data',
          )
QC_plot

In [ ]:
order = [
    "T1_control", "T2_control", "T3_control",
    "T1_rev", "T2_rev", "T3_rev", "T3_naive_rev",
    "T2_mix", "T3_mix_good", "T3_mix_bad",
]

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (5, 3), 
    "figure.dpi": (600)
}):
    sc.pl.violin(adata_sub, ['total_counts', 'n_genes_by_counts', 'pct_counts_mt'],
                 jitter=0.4, order=order, groupby = 'sample', rotation= 90, size = 0.5, palette = sample_colors, save = "sub_QC_violin.pdf")   

In [ ]:
filtered_marker_genes = {
    "Naive hPSC": {"KLF17", "DPPA5", "IL6ST", "ZNF729"},
    "ICM": {"ATG2A",  "ESRRB", "LAMA4", "EPHA4"},
    "Epiblast": {"POU5F1", "NANOG", "NODAL", "KLF17"},
    "Trophectoderm": {"GATA3", "CDX2", "TFAP2C", "KRT7"},
    "Hypoblast": {"PDGFRA", "GATA6", "GATA4", "SOX17"},
    
}

In [ ]:
filtered_marker_genes = marker_genes

In [ ]:
# plt.rcParams.update({'font.size': 4})
# plt.rcParams['ytick.labelsize'] = 5

# FIGSIZE=(3,3.5)
# #rcParams['figure.figsize']=FIGSIZE
# sc.set_figure_params(scanpy=True, fontsize=9)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
}):
    # Make Axes
    # Number of needed rows and columns (based on the row with the most columns)
    nrow=len(filtered_marker_genes)
    ncol=max([len(vs) for vs in filtered_marker_genes.values()])
    fig,axs=plt.subplots(nrow,ncol,figsize=(2.5*ncol,2*nrow))
    # Plot expression for every marker on the corresponding Axes object
    for row_idx,(cell_type,markers) in enumerate(filtered_marker_genes.items()):
        col_idx=0
        for marker in markers:
            ax=axs[row_idx,col_idx]
            sc.pl.umap(adata_sub,color=marker, size = 10, ax=ax,show=False,frameon=False, cmap="RdPu")
            
            # Add cell type as row label - here we simply add it as ylabel of
            # the first Axes object in the row
            if col_idx==0:
                # We disabled axis drawing in UMAP to have plots without background and border
                # so we need to re-enable axis to plot the ylabel
                ax.axis('on')
                ax.set(xlabel=None)
                ax.tick_params(
                    top='off', bottom='off', left='off', right='off', 
                    labelleft='off', labelbottom='off')
                ax.set_ylabel(cell_type+'\n', rotation=90)
                ax.set(frame_on=False)
            col_idx+=1
        # Remove unused column Axes in the current row
        while col_idx<ncol:
            axs[row_idx,col_idx].remove()
            col_idx+=1

# Alignment within the Figure
fig.tight_layout()
fig.savefig(output_dir +'/'+'umaps.pdf')

In [ ]:
# 1) Run DE for all clusters vs rest
sc.tl.rank_genes_groups(
    adata_sub,
    groupby= "integrated_celltype",
    method="wilcoxon"
)

In [ ]:
# 2) Heatmap of top N DE genes per cluster

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.dpi": (300),
    "axes.linewidth": 0.5
}):
    sc.pl.rank_genes_groups_heatmap(
        adata_sub,
        n_genes=10,        # number of top genes per cluster
        standard_scale="var",  # z-score per gene
        #swap_axes=True,        # genes on y, clusters on x (often nicer)
        show_gene_labels=True,
        dendrogram=False,
        figsize=(6, 2),
        var_group_positions=None,     
        cmap='viridis',
        save = "sub_DE_heatmap.pdf"
    )

## Save

In [ ]:
# group all epiblast together
adata_sub.obs["celltype_coarse"] = adata_sub.obs["integrated_celltype"].astype(str)
mask = adata_sub.obs["integrated_celltype"].str.contains("EPI", na=False)
adata_sub.obs.loc[mask, "celltype_coarse"] = "EPI"

In [ ]:
#save 
adata_sub.obs['scrublet_prediction'] = adata_sub.obs['scrublet_prediction'].astype(str)
adata_sub.write_h5ad(os.path.join(output_dir, "integrated_sub_adata.h5ad"))

In [ ]:
adata_sub.obs['cell.ID_']

In [ ]:
adata_sub.obs.to_csv(output_dir+"/meta_integrated_sub.csv")

In [ ]:
#save GEM for running EEPT
import pandas as pd
from scipy import sparse

X = adata_sub.layers["counts"]  # or adata_sub.raw.X if you stored raw there

if sparse.issparse(X):
    X = X.toarray()

GEM = pd.DataFrame(
    X,
    index=adata_sub.obs['cell.ID_'],          # cells
    columns=adata_sub.var_names         # or gene_symbols if you have them
).T                                     # genes × cells

GEM.index.name = "Gene"

GEM.to_csv(output_dir+"/GEM_forEEPT.csv")

In [ ]:
adata_sub.obs["celltype_fine"] = "other"

In [ ]:
adata_sub = sc.read_h5ad(os.path.join(output_dir, "integrated_sub_adata.h5ad"))

# Subset unspecified

In [ ]:
adata_unspecified = adata_sub[adata_sub.obs["integrated_celltype"].isin(["unspecified"])].copy()

## HVG, PCA, Cluster

In [ ]:
# Highly variable genes
sc.pp.highly_variable_genes(
    adata_unspecified,
    flavor = 'seurat_v3',
    n_top_genes = 2000,
    subset = False, 
    layer='counts' 
)

bdata_sub = adata_unspecified[:, adata_unspecified.var["highly_variable"]].copy()


sc.pp.regress_out(bdata_sub, ["pct_counts_mt", "n_genes_by_counts"])
sc.pp.scale(bdata_sub, max_value=10)

#compute PCA
sc.tl.pca(bdata_sub,
          n_comps = 100,
          svd_solver = 'arpack')


#fill NaNs with False so that subsetting to HVGs is possible
bdata_sub.var['highly_variable'].fillna(value = False,
                                    inplace = True)

adata_unspecified.obsm['X_pca'] = bdata_sub.obsm['X_pca'].copy()
adata_unspecified.uns['pca'] = bdata_sub.uns['pca'].copy()

adata_unspecified.varm['PCs'] = np.zeros(shape=(adata_unspecified.n_vars,
                                    100))
adata_unspecified.varm['PCs'][adata_unspecified.var['highly_variable']] = bdata_sub.varm['PCs']

sc.pl.pca_variance_ratio(adata_unspecified,
                         n_pcs = 100,
                         log = True)


In [ ]:
# Neighbors / Leiden / UMAP on final rep
sc.pp.neighbors(adata_unspecified, n_pcs = 30, n_neighbors=15, random_state=0)

sc.tl.umap(adata_unspecified, min_dist=0.3, spread=1.0)
sc.tl.leiden(adata_unspecified, resolution=0.5, key_added="leiden")


In [ ]:
sc.tl.leiden(adata_unspecified, resolution=0.2, key_added="leiden")

In [ ]:
idx = np.random.permutation(adata_unspecified.n_obs)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (300),
    'axes.titlesize':8,
     "legend.fontsize": 8,
    'axes.labelsize' :8
}):
    QC_plot = sc.pl.umap(adata_unspecified[idx, :],
           color = ['leiden', 'sample',  'sample_celltype', 'State', 'timepoint', "Condition", "n_genes_by_counts",  "pct_counts_mt",  'scrublet_cluster_score', 'phase'],
           wspace = 0.4,
           ncols = 4,
           alpha = 0.75,
           size = 5,
           save = "_subset_unspecified_QC.pdf",     # any continuous obs
    color_map="viridis"
#            legend_loc = 'on data',
          )
QC_plot

In [ ]:
marker_genes = {
    "Naive hPSC": {"KLF17", "DPPA5", "DNMT3L", "GATA6", "TBX3", "IL6ST", "DPPA3", "KLF5", "HORMAD1", "KHDC3L", "ALPP", "ZNF729"},
    "Primed hPSC": {"CD24", "ZIC2", "SFRP2"},
    "ICM": {"IFITM1", "GDF3", "ARGFX", "ATG2A", "MAGEA4", "ESRRB", "FGF1", "PRSS3", "LAMA4", "CCR8", "EPHA4", "GSC"},
    "Trophectoderm": {"GATA2", "GATA3", "CDX2", "TFAP2C", "KRT7", "KRT18", "OVOL1"},
    "Mural TE": {"HAND1"},
    "Polar TE": {"SDC1", "GCM1", "OVOL1", "CCR7", "CYP19A1", "DLX5", "MUC15"},
    "Epiblast": {"POU5F1", "GDF3", "ARGFX", "NANOG", "SOX2", "KLF17", "TDGF1"},
    "Early Epiblast": {"DPPA5", "ARGFX", "SOX2"},
    "Late Epiblast": {"FGF2", "TDGF1", "NODAL"},
    "Hypoblast": {"PDGFRA", "GATA6", "GATA4", "SOX17", "COL4A1", "APOA1", "RSPO3"},
    "Amnion": {"GABRP", "ISL1"},
    "Meso": {"APLNR", "CRABP2"}
}


In [ ]:
sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))

sc.pl.dotplot(
    adata_unspecified,
    groupby="leiden",
    var_names=marker_genes,  cmap=grey_to_pink,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "unspecified_marker.pdf")

In [ ]:
#based on the visual inspection of UMAP, annotate clusters:
cl_annotation = {str(i): "unspecified_1" for i in range(0,30)}  # Initialize all as "unknown"

# Assign specific values
for key in ["0", "1", "2"]:
    cl_annotation[key] = "unspecified_1"

for key in ["3"]:
    cl_annotation[key] = "unspecified_2"
    
for key in ["4"]:
    cl_annotation[key] = "unspecified_3"
    
    

adata_unspecified.obs["celltype_fine"] = adata_unspecified.obs.leiden.map(cl_annotation)

In [ ]:
idx = np.random.permutation(adata_unspecified.n_obs)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
    'axes.titlesize':8,
     "legend.fontsize": 8,
    'axes.labelsize' :8
}):
    QC_plot = sc.pl.umap(adata_unspecified[idx, :],
           color = ['leiden', "celltype_fine"],
           wspace = 0.4,
           ncols = 4,
           alpha = 0.75,
           size = 5,
           save = "_unspecified_finecelltype.pdf",     # any continuous obs
    color_map="viridis"
#            legend_loc = 'on data',
          )
QC_plot

In [ ]:
# 1) Run DE for all clusters vs rest
sc.tl.rank_genes_groups(
     adata_unspecified,
    groupby= "celltype_fine",
    method="wilcoxon"
)

In [ ]:
#Adjust these specific keys to control font sizes
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.dpi": 100,
    "axes.linewidth": 0.5 ,
    'axes.titlesize':7,
     "legend.fontsize": 7,
    'axes.labelsize' :7
}):
    sc.pl.rank_genes_groups_heatmap(
        adata_unspecified,
        n_genes=6,
        standard_scale="var",
        show_gene_labels=True,
        dendrogram=False,
        figsize=(9, 2), # Small figure size requires small fonts
        cmap='viridis',
        save="unspecified_DE_heatmap.pdf"
    )

In [ ]:
sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))

sc.pl.dotplot(
    adata_unspecified,
    groupby="celltype_fine",
    var_names=marker_genes,  cmap=grey_to_pink,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "unspecified_marker.pdf")

In [ ]:
adata_sub.obs["celltype_fine"] = adata_sub.obs["celltype_fine"].astype("str")
adata_sub.obs.loc[adata_unspecified.obs_names, "celltype_fine"] = adata_unspecified.obs["celltype_fine"]

# Subset trophoblast

In [ ]:
adata_TE = adata_sub[adata_sub.obs["celltype_coarse"].isin(["TE"])].copy()

## HVG, PCA, Cluster

In [ ]:
# Highly variable genes
sc.pp.highly_variable_genes(
    adata_TE,
    flavor = 'seurat_v3',
    n_top_genes = 2000,
    subset = False, 
    layer='counts' 
)

bdata_sub = adata_TE[:, adata_TE.var["highly_variable"]].copy()


sc.pp.regress_out(bdata_sub, ["pct_counts_mt", "n_genes_by_counts"])
sc.pp.scale(bdata_sub, max_value=10)

#compute PCA
sc.tl.pca(bdata_sub,
          n_comps = 100,
          svd_solver = 'arpack')


#fill NaNs with False so that subsetting to HVGs is possible
bdata_sub.var['highly_variable'].fillna(value = False,
                                    inplace = True)

adata_TE.obsm['X_pca'] = bdata_sub.obsm['X_pca'].copy()
adata_TE.uns['pca'] = bdata_sub.uns['pca'].copy()

adata_TE.varm['PCs'] = np.zeros(shape=(adata_TE.n_vars,
                                    100))
adata_TE.varm['PCs'][adata_TE.var['highly_variable']] = bdata_sub.varm['PCs']

sc.pl.pca_variance_ratio(adata_TE,
                         n_pcs = 100,
                         log = True)


In [ ]:
# Neighbors / Leiden / UMAP on final rep
sc.pp.neighbors(adata_TE, n_pcs = 30, n_neighbors=15, random_state=0)

sc.tl.umap(adata_TE, min_dist=0.3, spread=1.0)
sc.tl.leiden(adata_TE, resolution=0.5, key_added="leiden")


In [ ]:
idx = np.random.permutation(adata_TE.n_obs)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": 300,
    'axes.titlesize':8,
     "legend.fontsize": 8,
    'axes.labelsize' :8
}):
    QC_plot = sc.pl.umap(adata_TE[idx, :],
           color = ['leiden', 'sample',  'sample_celltype', 'State', 'timepoint', "Condition", "n_genes_by_counts",  "pct_counts_mt",  'scrublet_cluster_score', 'phase'],
           wspace = 0.4,
           ncols = 4,
           alpha = 0.75,
           size = 10,
           save = "_subset_TE_QC.pdf",     # any continuous obs
    color_map="viridis"
#            legend_loc = 'on data',
          )
QC_plot

In [ ]:
marker_genes = {
    # "Naive hPSC": {"KLF17", "DPPA5", "DNMT3L", "GATA6", "TBX3", "IL6ST", "DPPA3", "KLF5", "HORMAD1", "KHDC3L", "ALPP", "ZNF729"},
    # "Primed hPSC": {"CD24", "ZIC2", "SFRP2"},
    # "ICM": {"IFITM1", "GDF3", "ARGFX", "ATG2A", "MAGEA4", "ESRRB", "FGF1", "PRSS3", "LAMA4", "CCR8", "EPHA4", "GSC"},
    "Trophectoderm": {"GATA2", "GATA3", "CDX2", "TFAP2C", "KRT7", "KRT18", "OVOL1"},
    "Mural TE": {"HAND1"},
    "Polar TE": {"SDC1", "GCM1", "OVOL1", "CCR7", "CYP19A1", "DLX5", "MUC15"},
    "Epiblast": {"POU5F1", "GDF3", "ARGFX", "NANOG", "SOX2", "KLF17", "TDGF1"},
    "Early Epiblast": {"DPPA5", "ARGFX", "SOX2"},
    "Late Epiblast": {"FGF2", "TDGF1", "NODAL"},
    "Hypoblast": {"PDGFRA", "GATA6", "GATA4", "SOX17", "COL4A1", "APOA1", "RSPO3"},
    "Amnion": {"GABRP", "ISL1"},
    "Meso": {"APLNR", "CRABP2"}
}


In [ ]:
sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))

sc.pl.dotplot(
    adata_TE,
    groupby="leiden",
    var_names=marker_genes,  cmap=grey_to_pink,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "TE_marker.pdf")

In [ ]:
#based on the visual inspection of UMAP, annotate clusters:
cl_annotation = {str(i): "TE_1" for i in range(0,30)}  # Initialize all as "unknown"

# Assign specific values
for key in ["4", "0"]:
    cl_annotation[key] = "TE_2"

for key in ["7"]:
    cl_annotation[key] = "TE_3"
    
for key in ["6"]:
    cl_annotation[key] = "TE_4"
    

adata_TE.obs["celltype_fine"] = adata_TE.obs.leiden.map(cl_annotation)

In [ ]:
idx = np.random.permutation(adata_TE.n_obs)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
    'axes.titlesize':8,
     "legend.fontsize": 8,
    'axes.labelsize' :8
}):
    QC_plot = sc.pl.umap(adata_TE[idx, :],
           color = ['leiden', "celltype_fine"],
           wspace = 0.4,
           ncols = 4,
           alpha = 0.75,
           size = 10,
           save = "_TE_finecelltype.pdf",     # any continuous obs
    color_map="viridis"
#            legend_loc = 'on data',
          )
QC_plot

In [ ]:
sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))

sc.pl.dotplot(
    adata_TE,
    groupby="celltype_fine",
    var_names=marker_genes,  cmap=grey_to_pink,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "TE_marker.pdf")

In [ ]:
# 1) Run DE for all clusters vs rest
sc.tl.rank_genes_groups(
     adata_TE,
    groupby= "celltype_fine",
    method="wilcoxon"
)

In [ ]:
#Adjust these specific keys to control font sizes
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.dpi": 100,
    "axes.linewidth": 0.5 ,
    'axes.titlesize':7,
     "legend.fontsize": 7,
    'axes.labelsize' :7
}):
    sc.pl.rank_genes_groups_heatmap(
        adata_TE,
        n_genes=6,
        standard_scale="var",
        show_gene_labels=True,
        dendrogram=False,
        figsize=(9, 2), # Small figure size requires small fonts
        cmap='viridis',
        save="TE_DE_heatmap.pdf"
    )

In [ ]:
adata_sub.obs["celltype_fine"] = adata_sub.obs["celltype_fine"].astype(str)
adata_sub.obs.loc[adata_TE.obs_names, "celltype_fine"] = adata_TE.obs["celltype_fine"]
#adata_sub.obs["celltype_fine"] = adata_sub.obs["celltype_fine"].astype("category")

# Subset epiblast

In [ ]:
adata_EPI = adata_sub[adata_sub.obs["celltype_coarse"].isin(["EPI"])].copy()

## HVG, PCA, Cluster

In [ ]:
# Highly variable genes
sc.pp.highly_variable_genes(
    adata_EPI,
    flavor = 'seurat_v3',
    n_top_genes = 2000,
    subset = False, 
    layer='counts' 
)

bdata_sub = adata_EPI[:, adata_EPI.var["highly_variable"]].copy()


sc.pp.regress_out(bdata_sub, ["pct_counts_mt", "n_genes_by_counts"])
sc.pp.scale(bdata_sub, max_value=10)

#compuEPI PCA
sc.tl.pca(bdata_sub,
          n_comps = 100,
          svd_solver = 'arpack')


#fill NaNs with False so that subsetting to HVGs is possible
bdata_sub.var['highly_variable'].fillna(value = False,
                                    inplace = True)

adata_EPI.obsm['X_pca'] = bdata_sub.obsm['X_pca'].copy()
adata_EPI.uns['pca'] = bdata_sub.uns['pca'].copy()

adata_EPI.varm['PCs'] = np.zeros(shape=(adata_EPI.n_vars,
                                    100))
adata_EPI.varm['PCs'][adata_EPI.var['highly_variable']] = bdata_sub.varm['PCs']

sc.pl.pca_variance_ratio(adata_EPI,
                         n_pcs = 100,
                         log = True)


In [ ]:
# Neighbors / Leiden / UMAP on final rep
sc.pp.neighbors(adata_EPI, n_pcs = 30, n_neighbors=15, random_state=0)

sc.tl.umap(adata_EPI, min_dist=0.3, spread=1.0)
sc.tl.leiden(adata_EPI, resolution=0.5, key_added="leiden")


In [ ]:
adata_EPI

In [ ]:
idx = np.random.permutation(adata_EPI.n_obs)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": (100),
    'axes.titlesize':8,
     "legend.fontsize": 8,
    'axes.labelsize' :8
}):
    QC_plot = sc.pl.umap(adata_EPI[idx, :],
           color = ['leiden', 'sample',  'sample_celltype', 'State', 'timepoint', "Condition", "n_genes_by_counts",  "pct_counts_mt",  'scrublet_cluster_score', 'phase'],
           wspace = 0.4,
           ncols = 4,
           alpha = 0.75,
           size = 10,
           save = "_subset_EPI_QC.pdf",     # any continuous obs
    color_map="viridis"
#            legend_loc = 'on data',
          )
QC_plot

In [ ]:
marker_genes = {
    "Naive hPSC": {"KLF17", "DPPA5", "DNMT3L", "GATA6", "TBX3", "IL6ST", "DPPA3", "KLF5", "HORMAD1", "KHDC3L", "ALPP", "ZNF729"},
    "Primed hPSC": {"CD24", "ZIC2", "SFRP2"},
    "ICM": {"IFITM1", "GDF3", "ARGFX", "ATG2A", "MAGEA4", "ESRRB", "FGF1", "PRSS3", "LAMA4", "CCR8", "EPHA4", "GSC"},
    "Trophectoderm": {"GATA2", "GATA3", "CDX2", "TFAP2C", "KRT7", "KRT18", "OVOL1"},
    "Mural TE": {"HAND1"},
    "Polar TE": {"SDC1", "GCM1", "OVOL1", "CCR7", "CYP19A1", "DLX5", "MUC15"},
    "Epiblast": {"POU5F1", "GDF3", "ARGFX", "NANOG", "SOX2", "KLF17", "TDGF1"},
    "Early Epiblast": {"DPPA5", "ARGFX", "SOX2"},
    "Late Epiblast": {"FGF2", "TDGF1", "NODAL"},
    "Hypoblast": {"PDGFRA", "GATA6", "GATA4", "SOX17", "COL4A1", "APOA1", "RSPO3"},
    "Amnion": {"GABRP", "ISL1"},
    "Meso": {"APLNR", "CRABP2"}
}


In [ ]:
sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))

sc.pl.dotplot(
    adata_EPI,
    groupby="leiden",
    var_names=marker_genes,  cmap=grey_to_pink,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "EPI_marker.pdf")

In [ ]:
#based on the visual inspection of UMAP, annotate clusters:
cl_annotation = {str(i): "EPI_1" for i in range(0,30)}  # Initialize all as "unknown"

# Assign specific values
for key in ["4"]:
    cl_annotation[key] = "EPI_2"

for key in ["2"]:
    cl_annotation[key] = "EPI_3"
    


adata_EPI.obs["celltype_fine"] = adata_EPI.obs.leiden.map(cl_annotation)

In [ ]:
idx = np.random.permutation(adata_EPI.n_obs)

with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.figsize": (3, 3), 
    "figure.dpi": 100,
    'axes.titlesize':8,
     "legend.fontsize": 8,
    'axes.labelsize' :8
}):
    QC_plot = sc.pl.umap(adata_EPI[idx, :],
           color = ['leiden', "celltype_fine"],
           wspace = 0.4,
           ncols = 4,
           alpha = 0.75,
           size = 10,
           save = "_epi_finecelltype.pdf",     # any continuous obs
    color_map="viridis"
#            legend_loc = 'on data',
          )
QC_plot

In [ ]:
sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))

sc.pl.dotplot(
    adata_EPI,
    groupby="celltype_fine",
    var_names=marker_genes,  cmap=grey_to_pink,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "EPI_marker.pdf")

In [ ]:
# 1) Run DE for all clusters vs rest
sc.tl.rank_genes_groups(
     adata_EPI,
    groupby= "celltype_fine",
    method="wilcoxon"
)

In [ ]:
#Adjust these specific keys to control font sizes
with plt.rc_context({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "figure.dpi": 100,
    "axes.linewidth": 0.5 ,
    'axes.titlesize':7,
     "legend.fontsize": 7,
    'axes.labelsize' :7
}):
    sc.pl.rank_genes_groups_heatmap(
        adata_EPI,
        n_genes=6,
        standard_scale="var",
        show_gene_labels=True,
        dendrogram=False,
        figsize=(9, 2), # Small figure size requires small fonts
        cmap='viridis',
        save="EPI_DE_heatmap.pdf"
    )

In [ ]:
adata_sub.obs.loc[adata_EPI.obs_names, "celltype_fine"] = adata_EPI.obs["celltype_fine"]
#adata_sub.obs["celltype_fine"] = adata_sub.obs["celltype_fine"].astype("category")

# Save

In [ ]:
#visualise the clusters so far
#plot for QC
sc.settings.set_figure_params(dpi = 100)
sc.set_figure_params(figsize=(3, 3))

g=sc.pl.umap(adata_sub, color=["integrated_celltype","celltype_fine", "celltype_coarse"],
           #legend_loc="on data",
           ncols = 4,
           legend_fontsize = 'xx-small',
           size = 10,
           alpha = 0.75,
           wspace=0.5,
           save = "_integrated_fine_celltype.pdf")

g

In [ ]:
adata_sub.obs['scrublet_prediction'] = adata_sub.obs['scrublet_prediction'].astype(str)
adata_sub.write_h5ad(os.path.join(output_dir, "integrated_sub_adata.h5ad"))

In [ ]:
adata_sub.obs.to_csv(output_dir+"/meta_integrated_sub_fine.csv")

In [ ]:
output_dir+"/meta_integrated_sub_fine.csv"

# Proportions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Define your variables
group_by = "sample"        # The x-axis (e.g., Sample, Condition)
color_by = "celltype_fine"        # The composition (e.g., Cell Type, Cluster)

# 2. Calculate the cross-tabulation (counts)
crosstab = pd.crosstab(adata_sub.obs[group_by], adata_sub.obs[color_by])

# 3. Normalize to get ratios (rows sum to 1)
crosstab_norm = crosstab.div(crosstab.sum(1), axis=0)

# 4. Plot
with plt.rc_context({"figure.figsize": (6, 4), "figure.dpi": 300}):
    ax = crosstab_norm.plot(
        kind='bar', 
        stacked=True, 
        width=0.8, 
        edgecolor='none'
    )
    
    # Optional: Move legend outside
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
    
    plt.xlabel(group_by)
    plt.ylabel('Proportion of Cells')
    plt.title(f'Ratio of {color_by} per {group_by}')
    plt.gca().grid(False)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/ratio_{color_by}_by_{group_by}.pdf")
    plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Variables (Adjust these if needed) ---
group_by = "sample"        # The x-axis (e.g., Sample, Condition)
color_by = "celltype_coarse"        # The composition (Must match the UMAP color variable)
# ------------------------------------------

# 1. Calculate the cross-tabulation and normalize
crosstab = pd.crosstab(adata_sub.obs[group_by], adata_sub.obs[color_by])
crosstab_norm = crosstab.div(crosstab.sum(1), axis=0)

# 2. Retrieve the UMAP colors from adata.uns
umap_colors = None
if f"{color_by}_colors" in adata_sub.uns:
    umap_colors = adata_sub.uns[f"{color_by}_colors"]
else:
    print(f"Warning: UMAP colors for '{color_by}' not found in adata.uns.")

# 3. Plot, passing the colors list
with plt.rc_context({
    "figure.figsize": (6, 4), 
    "figure.dpi": 300,
    "axes.grid": False  # To ensure grid is off, as requested earlier
}):
    ax = crosstab_norm.plot(
        kind='bar', 
        stacked=True, 
        width=0.8, 
        edgecolor='none',
        # APPLY THE UMAP COLORS HERE
        color=umap_colors 
    )
    
    # Optional: Move legend outside
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
    
    plt.xlabel(group_by)
    plt.ylabel('Proportion of Cells')
    plt.title(f'Ratio of {color_by} per {group_by}')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/ratio_{color_by}_matched_colors.pdf")
    # plt.show()

# ExM?

In [ ]:
sys.path.append('/nfs/team292/rs40/resource')

import marker_genes_dict

In [ ]:
marker_genes = marker_genes_dict.markergenes_datasets['Tyser_DEG']
title = 'Tyser_DEG'

In [ ]:
filtered_marker_genes = {}
skipped_genes = {}

for category, genes in marker_genes.items():
    # Find genes that exist in the dataset
    existing_genes = [gene for gene in genes if gene in adata_sub.var_names]

    # Track skipped genes
    skipped_for_category = [gene for gene in genes if gene not in adata_sub.var_names]

    # Only add category if there are existing genes
    if existing_genes:
        filtered_marker_genes[category] = existing_genes

    # Store skipped genes if any
    if skipped_for_category:
        skipped_genes[category] = skipped_for_category

In [ ]:
# Print skipped genes
print("Skipped Genes:")
for category, genes in skipped_genes.items():
    print(f"{category}: {genes}")

In [ ]:
sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))

sc.pl.dotplot(
    adata_sub,
    groupby="celltype_fine",
    var_names=filtered_marker_genes,  cmap=grey_to_pink,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "Tyser_DEG_marker.pdf")

In [ ]:
marker_genes = marker_genes_dict.markergenes_datasets['Pham']
title = 'Pham'

In [ ]:
filtered_marker_genes = {}
skipped_genes = {}

for category, genes in marker_genes.items():
    # Find genes that exist in the dataset
    existing_genes = [gene for gene in genes if gene in adata_sub.var_names]

    # Track skipped genes
    skipped_for_category = [gene for gene in genes if gene not in adata_sub.var_names]

    # Only add category if there are existing genes
    if existing_genes:
        filtered_marker_genes[category] = existing_genes

    # Store skipped genes if any
    if skipped_for_category:
        skipped_genes[category] = skipped_for_category

In [ ]:
# Print skipped genes
print("Skipped Genes:")
for category, genes in skipped_genes.items():
    print(f"{category}: {genes}")

In [ ]:
sc.settings.set_figure_params(dpi = 150)
sc.set_figure_params(figsize=(3, 3))

sc.pl.dotplot(
    adata_sub,
    groupby="celltype_fine",
    var_names=filtered_marker_genes,  cmap=grey_to_pink,
    standard_scale="var",  # standard scale: normalize each gene to range from 0 to 1
    save = "Pham_marker.pdf")